# **Appendix A: Code**

# Practical 1 - Computing the Kaplan-Yorke Dimension

## Problem definition + Simulation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from tqdm import tqdm

## CONSTANTS

beta_1 = 1
beta_2 = 0.84
rho = 0.048

kappa_prime = 0.8544
kappa_cr_1 = beta_1 - 1
kappa_cr_2 = (
    kappa_cr_1 + rho*(2*beta_1 + (beta_2-1)*(2+rho)) + np.sqrt(
        4*rho**2 * (kappa_cr_1+beta_2) + (kappa_cr_1 + rho**2 * (beta_2-1))**2
    )
) / (2*(1+rho))

kappa = kappa_cr_2 * kappa_prime
v_0 = 1

## EOMS

def backup_func(t, input_vars):
    x, y, z = input_vars

    return (
        (-kappa * (np.exp(x) - v_0) + np.exp(x) * (
            x*(beta_1-1) + y - z
        ) + rho * np.exp(x) * (beta_2*x + z)),
        -kappa * (np.exp(x)-v_0),
        -rho*np.exp(x)*(
            beta_2*x+z
        )
    )

## alternate jacobian from lectures potentially instead?
#from scipy.differentiate import jacobian

## manual implementation
def jacobian(input_vars, kappa=kappa, beta_1=beta_1, beta_2=beta_2, rho=rho):

    assert len(input_vars) == 3

    x1, x2, x3 = input_vars

    ex = np.exp(x1)

    J11 = -kappa * ex + ex * (x1*(beta_1 - 1) + x2 - x3 + beta_1 - 1) + rho * ex * (beta_2 * x1 + x3 + beta_2)
    J12 = ex
    J13 = ex * (-1 + rho)

    J21 = -kappa * ex
    J22 = 0
    J23 = 0

    ## old version: J31 = -rho * ex * beta_2 * (x1 + 1)
    J31 = -rho * ex * (beta_2*x1 + x3 + beta_2)
    J32 = 0
    J33 = -rho * ex


    jac = np.array([
        [J11, J12, J13],
        [J21, J22, J23],
        [J31, J32, J33]
    ])

    return jac

### Part 1: Numerical Simulation

In [ ]:
t_start = 0
t_transient = 500
t_end = 4000
dt = 0.1
t_eval = np.arange(t_start, t_end+dt, dt)
X_0 = [0.05, 0, 0]

soln = solve_ivp(backup_func, (t_start, t_end), X_0, t_eval=t_eval, method='Radau', rtol=1e-10, atol=1e-8)

In [ ]:
n_points = 0

t = soln.t[-n_points:]
x, y, z = soln.y[:,-n_points:]

fig, ax = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
ax = ax.flatten()

ax[0].plot(t, x)
ax[0].set_title('$x_1$')
ax[1].plot(t, y)
ax[1].set_title('$x_2$')
ax[2].plot(t, z)
ax[2].set_title('$x_3$')

ax[2].set_xlabel('time')

ax_3d = fig.add_axes((0, -1, 1, 1), projection='3d')
ax_3d.plot(x, y, z)

ax_3d.set_xlabel('$x_1$')
ax_3d.set_ylabel('$x_2$')
ax_3d.set_zlabel('$x_3$')

fig.tight_layout()

#fig.suptitle('Numerical Simulation of the Quasi-Static Spring Slider', size=10)

### Part 2: Calculating the Lyapunov Exponents

We need to evolve the basis vectors then perform QR renormalisation at each step, collecting the logarithm of the absolute values of the diagonals of R at each step

In [ ]:
def extended_func(t, input_vars, dt=0.1):

    assert len(input_vars) == 12 # 3 input vars (x1, x2, x3) + 9 basis vars (3x3 matrix representing 3 basis vectors) 

    X = input_vars[:3]
    flat_basis = input_vars[3:]

    basis = flat_basis.reshape(3, 3)

    dXdt = backup_func(t, X)
    J = jacobian(X)

    A = np.eye(3) + J * dt

    new_basis = A @ basis

    return np.concatenate([dXdt, new_basis.flatten()])

In [ ]:
from scipy.integrate import solve_ivp

t_start = 0
t_transient=400
t_end = 4000
dt = 0.1

t_eval = np.arange(t_transient, t_end, dt)
X_0 = [0.05, 0, 0]
initial_basis = np.eye(3)

intial_vars = np.concatenate([X_0, initial_basis.flatten()])

lyap_history = []
cur_vars = intial_vars

lyap_total = np.zeros(3)

for cur_time in tqdm(np.arange(t_start, t_end, dt)):

    soln = solve_ivp(extended_func, (cur_time, cur_time+dt), cur_vars, method='RK45', rtol=1e-8, atol=1e-10)

    cur_vars = soln.y[:,-1]

    ## lyapunov collecting step
    cur_basis = cur_vars[3:].reshape(3, 3)

    Q, R = np.linalg.qr(cur_basis)

    new_basis = Q

    if cur_time >= t_transient:
        ## the r_ii values (the amount the bases stretch by)
        stretch = np.diag(R)

        ## collect Lyapunov exponents after the transient
        lyap_total += np.log(np.abs(stretch))

        lyap_exponents = np.log(np.abs(stretch)) / cur_time

        lyap_history.append(lyap_exponents)

        ## set the basis vectors to the orthogonal ones
        cur_vars[3:] = Q.flatten()        

print(lyap_total/(t_end-t_transient))

In [ ]:
from scipy.integrate import solve_ivp

t_start = 0
t_transient=4000
t_end = 10000
dt = 0.1

t_eval = np.arange(t_transient, t_end, dt)
X_0 = [0.05, 0, 0]
initial_basis = np.eye(3)

intial_vars = np.concatenate([X_0, initial_basis.flatten()])

lyap_history = []
cur_vars = intial_vars

lyap_total = np.zeros(3)

for cur_time in tqdm(np.arange(t_start, t_end, dt)):

    soln = solve_ivp(extended_func, (cur_time, cur_time+dt), cur_vars, method='RK45', rtol=1e-8, atol=1e-10)

    cur_vars = soln.y[:,-1]

    ## lyapunov collecting step
    cur_basis = cur_vars[3:].reshape(3, 3)

    Q, R = np.linalg.qr(cur_basis)

    new_basis = Q

    if cur_time >= t_transient:
        ## the r_ii values (the amount the bases stretch by)
        stretch = np.diag(R)

        ## collect Lyapunov exponents after the transient
        lyap_total += np.log(np.abs(stretch))

        lyap_exponents = np.log(np.abs(stretch)) / cur_time

        lyap_history.append(lyap_exponents)

        ## set the basis vectors to the orthogonal ones
        cur_vars[3:] = Q.flatten()        

print(lyap_total)

In [ ]:
lyap_total / t_end

In [ ]:
b = [[float(j) for j in list(i)] for i in lyap_history]

with open('exponents.txt', 'w') as outfile:
    outfile.write(repr(b))

In [ ]:
import matplotlib.pyplot as plt

a = np.array(lyap_history).T
final_le = a.sum(axis=1)

2+final_le[0]+final_le[1] / abs(final_le[2])

### Part 2.1: New Attempt -- Exponent from pre-computed system

In [ ]:
t_transient = 500

X = soln.y.T
t = soln.t

le_history = []

le_total = np.zeros(3)

initial_basis = np.eye(3)

V = initial_basis

for t_id in tqdm(range(len(t))):

    cur_time = t[t_id]
    cur_posn = X[t_id]

    if cur_time >= t_transient:

        dt = t[t_id] - t[t_id - 1]


        cur_time = t[t_id]
        cur_posn = X[t_id]

        jac = jacobian(cur_posn)

        A = np.eye(3) + jac * dt

        V_prime = A @ V

        Q, R = np.linalg.qr(V_prime)

        V = Q.copy()

        ## record lyapunov exponents
        R_diagonals = np.diag(R)

        le_current = np.log(np.abs(R_diagonals))

        le_total += le_current
        
        le_history.append(le_total.copy() / (cur_time - t_transient))

lyapunov_final = le_total / (t[-1] - t_transient)
lyapunov_final

In [ ]:
1/lyapunov_final[0]

In [ ]:
t = soln.t
lyap_t = t[t>=t_transient] 

l1, l2, l3 = np.transpose(le_history)

plt.plot(lyap_t, l1, label='$\\lambda_1$', color='black')
plt.plot(lyap_t, l2, label='$\\lambda_2$', color='lightblue')
plt.plot(lyap_t, l3, label='$\\lambda_3$', color='red')

plt.xlabel('Time')

plt.legend()

In [ ]:
final_le = lyapunov_final

print(f'Lyapunov Time: {1/final_le[0]}')
print(f'Two sum: {sum(final_le[:-1])}')
print(f'Three sum: {sum(final_le)}')
print(f'Kaplan-Yorke Dimension: {2+(final_le[0]+final_le[1])/np.abs(final_le[2])}')

# Practical 2 (Correlation Dimension)

## Part 1: Implementing the Theiler Window

### Finding the window size

In [ ]:
DEFAULT_NUM_PTS = 100

def autocorrelation(arr, num_pts = DEFAULT_NUM_PTS):
    '''
    Returns array of autocorrelation for an array
    '''

    corr = np.correlate(arr[:num_pts], arr[:num_pts], mode='full')

    return corr[num_pts-1:]

def sign_changes(arr):
    '''
    Returns indices before sign changes
    '''

    return np.where(np.diff(np.sign(arr)))[0]

t = soln.t

transient_mask = t >= t_transient

t = t[transient_mask]

x1, x2, x3 = soln.y[:,transient_mask]

x1_pts = len(x1)

ac1 = autocorrelation(x1, x1_pts)
ac2 = autocorrelation(x2, x1_pts)
ac3 = autocorrelation(x3, x1_pts)

dt = t[1]-t[0]
ac_x = t*dt

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(ac_x, ac1)
ax.plot(ac_x, ac2)
ax.plot(ac_x, ac3)

ax.set_xlabel('Time Units')
ax.set_ylabel('Autocorrelation')

z1 = sign_changes(ac1)
z2 = sign_changes(ac2)
z3 = sign_changes(ac3)

print(z1)
print(z2)
print(z3)

## choose first zero of the x window
all_zeros = np.concat([z1, z2, z3])
FIRST_ZERO = min(all_zeros)

display(f'First zero of autocorrelation function: {FIRST_ZERO} // {FIRST_ZERO*dt}')

## Part 2: The Correlation Sum
We now wish to implement a Theiler window of size `window_size` to calculate the Correlation sum. This is used for the initial calculation, but of the Correlation Dimension, but for repeated calculations (to investigate parameter dependence), the slightly modified code in the next section is used instead.

In [ ]:
## estimate inter-point distances
from scipy.spatial.distance import pdist

samples = soln.y.T[::100]
dists = pdist(samples)
min_dist = min(dists)
max_dist = max(dists)

display(f'{max_dist} // {min_dist}')

In [ ]:
## set epsilon range
## will do 0.005 * max_dist -> 0.1 * max_dist
## 30 log-spaced points

log_max_eps = np.log(0.1)
log_min_eps = np.log(0.0001)
num_eps = 100

log_eps_range = np.linspace(log_min_eps, log_max_eps, num_eps)
eps_range = np.exp(log_eps_range)

eps_range

In [ ]:
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist

## correlation sum

batch_size = 128
eps = 0.1
window_size = FIRST_ZERO

t = soln.t
timestep = t[1] - t[0]

pts = soln.y.T ### points in a (n_pts, n_dims) shape

def correlation_sum(eps, data, window = window_size, batch = batch_size):
    '''
    calculates the correlation sum, C(eps) assuming `data` is all the points up to and including time T with a Theiler window of size `window`
    '''

    ## assert data is 3-dimensional and in the correct orientation
    assert data.shape[1] == 3, 'Incorrect dimensions; data should be in shape `(n_points, n_dimensions)` // n_dimensions currently must be 3'

    tree = cKDTree(data)
    T = data.shape[0]

    scale_factor = 2 / ((T - window) * (T - window - 1))

    neighbours = 0

    for i_0 in tqdm(range(0, T - window - 1, batch)):

        i_1 = min(i_0+batch, T-window-1)

        x_i = data[i_0:i_1]

        ## all pts within eps of x_i
        idxs = np.array(tree.query_ball_point(x_i, eps))

        for row, vals in enumerate(idxs):
            i = i_0 + row
            idx = np.asarray(vals)
            valid_idxs = idx > (i + window)

            neighbours += np.sum(valid_idxs)

    return scale_factor * neighbours

def correlation_sum_NO_THEILER(eps, data):

    assert data.shape[1] == 3, 'Incorrect dimensions; data should be in shape `(n_points, n_dimensions)` // n_dimensions currently must be 3'

    tree = cKDTree(data)
    T = data.shape[0]

    scale_factor = 2 / (T**2)
    neighbours = 0

    for i_0 in tqdm(range(T)):

        x_i = data[i_0]
        neighbours += len(tree.query_ball_point(x_i, eps))

    return scale_factor * neighbours

In [ ]:
corr_sums = []

for e_id, eps in enumerate(eps_range):
    print(f'{e_id}: epsilon={eps}')

    corr_sums.append(correlation_sum_NO_THEILER(eps, pts))

In [ ]:
## plotting log-log graph
with open('correlation_sums-NO_THEILER.txt', 'w') as corr_file:

    corr_file.write(repr(list(eps_range)))
    corr_file.write('\n')
    corr_file.write(repr(corr_sums))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FROM_FILE = True
THEILER = False

if THEILER:
    filepath = 'correlation_sums.txt' 
else:
    filepath = 'correlation_sums-NO_THEILER.txt'

if FROM_FILE:
    
    with open(filepath, 'r') as corr_file:

        eps, corr = corr_file.read().split('\n')

        eps_range = eval(eps)
        corr_sums = eval(corr)

else:
    eps_range = [np.float64(0.004472945488587768), np.float64(0.004765763771385856), np.float64(0.005077751200546132), np.float64(0.005410162670977284), np.float64(0.005764335228415282), np.float64(0.006141693447370463), np.float64(0.006543755161138205), np.float64(0.0069721375669207965), np.float64(0.0074285637306168726), np.float64(0.007914869517442406), np.float64(0.008433010976259994), np.float64(0.008985072207318172), np.float64(0.009573273745046818), np.float64(0.010199981489626484), np.float64(0.01086771622325671), np.float64(0.01157916374940031), np.float64(0.012337185695786198), np.float64(0.013144831024623379), np.float64(0.01400534829632313), np.float64(0.014922198736057256), np.float64(0.015899070155709558), np.float64(0.016939891787218213), np.float64(0.018048850086972674), np.float64(0.019230405574834453), np.float64(0.020489310775512783), np.float64(0.021830629334460002), np.float64(0.02325975638517584), np.float64(0.024782440249843156), np.float64(0.026404805560580814), np.float64(0.028133377894313374), np.float64(0.029975110020345495), np.float64(0.03193740986621564), np.float64(0.03402817031431505), np.float64(0.036255800949121765), np.float64(0.038629261882745396), np.float64(0.04115809979483774), np.float64(0.04385248633183125), np.float64(0.046723259019957095), np.float64(0.049781964856605625), np.float64(0.05304090675536514), np.float64(0.056513193031552965), np.float64(0.060212790127282555), np.float64(0.06415457878814086), np.float64(0.06835441391743263), np.float64(0.07282918834874183), np.float64(0.07759690079331942), np.float64(0.08267672823560004), np.float64(0.08808910306804096), np.float64(0.09385579527554012), np.float64(0.10000000000000002)]
    corr_sums = [np.float64(3.489857927839813e-05), np.float64(3.966300675222325e-05), np.float64(4.49725050633028e-05), np.float64(5.105288478992598e-05), np.float64(5.787628047996371e-05), np.float64(6.548169121439123e-05), np.float64(7.410176841447533e-05), np.float64(8.390098100780485e-05), np.float64(9.481503762239553e-05), np.float64(0.00010735752866579922), np.float64(0.00012162584514840757), np.float64(0.00013722679549907504), np.float64(0.0001547973814430711), np.float64(0.0001748499005313848), np.float64(0.00019728844289026927), np.float64(0.00022232304122714598), np.float64(0.0002503869280105892), np.float64(0.0002822366377138977), np.float64(0.00031811044505678594), np.float64(0.0003588822457636369), np.float64(0.00040494784222273254), np.float64(0.0004572476055793907), np.float64(0.0005162319268936543), np.float64(0.0005820080379481985), np.float64(0.0006559659352760506), np.float64(0.0007396047676497494), np.float64(0.0008343200168287097), np.float64(0.0009409719972674031), np.float64(0.0010603016036403207), np.float64(0.0011943496539637748), np.float64(0.0013453664717772698), np.float64(0.0015135883862396732), np.float64(0.0017019168310699001), np.float64(0.001912373783724062), np.float64(0.0021489626059287125), np.float64(0.0024144262372297787), np.float64(0.0027117735565129436), np.float64(0.003043715909926381), np.float64(0.0034142987962789873), np.float64(0.0038282838062913523), np.float64(0.00428957778932002), np.float64(0.004802143074585686), np.float64(0.005368270616469538), np.float64(0.0060040232289152815), np.float64(0.006714664695830097), np.float64(0.007502234956589916), np.float64(0.00836816532892975), np.float64(0.009317072239627003), np.float64(0.010358280134220355), np.float64(0.011502927411318042)]

log_eps = np.log(eps_range)
log_corr = np.log(corr_sums)

fig, axs = plt.subplots(2, 2, figsize=(20, 20))
axs = axs.flatten()

min_length = 10
llim_options = np.arange(0, 100-min_length, 1)

for llim in llim_options:
    lim_range = range(min_length+llim, len(eps_range))

    grads = []
    for lim in lim_range:
        log_eps_range = log_eps[llim:lim]
        log_corr_sums = log_corr[llim:lim]
        coeffs = np.polyfit(log_eps_range, log_corr_sums, deg=1)

        grads.append(coeffs[0])

    axs[0].scatter(np.log(eps_range), np.log(corr_sums))
    #axs[0].plot(log_eps_range, coeffs[0]*log_eps_range + coeffs[1])

    axs[1].scatter(np.array(lim_range), grads, marker='o', label=f'{llim}', color='blue', alpha=0.5)

axs[1].set_xlabel('Upper Limit')

## WITH Theiler Window
# dim_guess = 2.38

## WITHOUT Theiler Window
dim_guess = 1.765

axs[1].hlines(dim_guess, min_length, 100, color='black', linewidth=2)

vals = np.divide(log_corr, log_eps)

axs[2].plot(eps_range, vals)

log_eps0 = log_eps[0]
log_corr0 = log_corr[0]

d_corr = []

eps_min_id = 1
eps_max_id = len(log_eps)

for i in range(eps_min_id, eps_max_id):

    delta_log_eps = log_eps[i] - log_eps0
    delta_log_corr = log_corr[i] - log_corr0

    d_corr.append(delta_log_corr / delta_log_eps)

plot_min_eps_id = 0
axs[3].plot(eps_range[eps_min_id:eps_max_id][plot_min_eps_id:], d_corr[plot_min_eps_id:])

print(max(d_corr[plot_min_eps_id:]))

# Effect of Changing Sampling Parameters

## Bonus Simulation code so I don't have to scroll up as far

In [ ]:
t_start = 0
t_transient = 400
t_end = 20000
dt = 0.01
t_eval = np.arange(t_start, t_end+dt, dt)
X_0 = [0.05, 0, 0]

soln = solve_ivp(backup_func, (t_start, t_end), X_0, t_eval=t_eval, method='Radau', rtol=1e-8, atol=1e-10)

## Lyapunov Dimension Code

In [ ]:
def solve_le(X, t, t_transient=500):
    le_history = []

    le_total = np.zeros(3)

    initial_basis = np.eye(3)

    V = initial_basis

    for t_id in tqdm(range(len(t))):

        cur_time = t[t_id]
        cur_posn = X[t_id]

        if cur_time >= t_transient:

            dt = t[t_id] - t[t_id - 1]


            cur_time = t[t_id]
            cur_posn = X[t_id]

            jac = jacobian(cur_posn)

            A = np.eye(3) + jac * dt

            V_prime = A @ V

            Q, R = np.linalg.qr(V_prime)

            V = Q.copy()

            ## record lyapunov exponents
            R_diagonals = np.diag(R)

            le_current = np.log(np.abs(R_diagonals))

            le_total += le_current
            
            le_history.append(le_total.copy() / (cur_time-t_transient))

    lyapunov_final = le_total / (t[-1] - t_transient)
    
    return lyapunov_final, le_history

def ky_dim(les):

    assert len(les) == 3, 'Wrong number of exponents dingus'
    assert les[0] + les[1] >= 0, 'bad first sum'
    assert les[0] + les[1] + les[2] < 0, 'bad second sum'

    return 2 + (les[0]+les[1])/np.abs(les[2])

## **Trajectory Length**

In [ ]:
def solve_le(X, t, t_transient=500, track = True):
    le_history = []

    le_total = np.zeros(3)

    initial_basis = np.eye(3)

    V = initial_basis

    loop_iter = tqdm(range(len(t))) if track else range(len(t))

    for t_id in loop_iter:

        cur_time = t[t_id]
        cur_posn = X[t_id]

        if cur_time >= t_transient:

            dt = t[t_id] - t[t_id - 1]

            jac = jacobian(cur_posn)

            A = np.eye(3) + jac * dt

            V_prime = A @ V

            Q, R = np.linalg.qr(V_prime)

            V = Q.copy()

            ## record lyapunov exponents
            R_diagonals = np.diag(R)

            le_current = np.log(np.abs(R_diagonals))

            le_total += le_current
            
            le_history.append(le_total.copy())

    le_times = t[t >= t_transient]
    

    lyapunov_final = le_total / (t[-1] - t_transient)
    
    return lyapunov_final, np.divide(np.transpose(le_history), le_times)

def ky_dim(les):

    assert len(les) == 3, 'Wrong number of exponents dingus'
    assert (les[0] + les[1] >= 0).any(), 'bad first sum'
    assert (les[0] + les[1] + les[2] < 0).any(), 'bad second sum'

    return 2 + (les[0]+les[1])/np.abs(les[2])

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist

DEFAULT_NUM_PTS = 10000

def autocorrelation(arr, num_pts = DEFAULT_NUM_PTS):
    '''
    Returns array of autocorrelation for an array
    '''

    corr = np.correlate(arr[:num_pts], arr[:num_pts], mode='full')

    return corr[num_pts-1:]

def sign_changes(arr):
    '''
    Returns indices before sign changes
    '''

    return np.where(np.diff(np.sign(arr)))[0]

## correlation sum

batch_size = 128
eps = 0.1
window_size = 1

def correlation_sum(eps, data, window = window_size, batch = batch_size, track=True):
    '''
    calculates the correlation sum, C(eps) assuming `data` is all the points up to and including time T with a Theiler window of size `window`
    '''

    ## assert data is 3-dimensional and in the correct orientation
    assert data.shape[1] == 3, 'Incorrect dimensions; data should be in shape `(n_points, n_dimensions)` // n_dimensions currently must be 3'

    tree = cKDTree(data)
    T = data.shape[0]

    scale_factor = 2 / ((T - window) * (T - window - 1))

    neighbours = 0

    loop_iter = tqdm(range(0, T - window - 1, batch)) if track else range(0, T - window - 1, batch)

    for i_0 in loop_iter:

        i_1 = min(i_0+batch, T-window-1)

        x_i = data[i_0:i_1]

        ## all pts within eps of x_i
        idxs = np.array(tree.query_ball_point(x_i, eps))

        for row, vals in enumerate(idxs):
            i = i_0 + row
            idx = np.asarray(vals)
            valid_idxs = idx > (i + window)

            neighbours += np.sum(valid_idxs)

    return scale_factor * neighbours

def correlation_sum_NO_THEILER(eps, data, window=None, track=True):

    assert data.shape[1] == 3, 'Incorrect dimensions; data should be in shape `(n_points, n_dimensions)` // n_dimensions currently must be 3'

    tree = cKDTree(data)
    T = data.shape[0]

    scale_factor = 2 / (T**2)
    neighbours = 0

    loop_iter = tqdm(range(T)) if track else range(T)

    for i_0 in loop_iter:

        x_i = data[i_0]
        neighbours += len(tree.query_ball_point(x_i, eps))

    return scale_factor * neighbours

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist

FIRST_ZERO = 148 

def get_eps_range(X, num_eps=20):
    '''
    Choose suitable epsilon range
    '''

    assert X.shape[1] == 3, 'Wrong dimensions' ## assert 3 dimensional

    num_pts = X.shape[0]

    sample_spacing = num_pts // 5000

    if sample_spacing < 1:
        sample_spacing = 1

    dists = pdist(X[::sample_spacing])

    min_dist = min(dists)
    max_dist = max(dists)

    lower_eps = np.floor(np.log(min_dist))
    upper_eps = np.floor(np.log(max_dist))-1

    log_eps_range = np.linspace(lower_eps, upper_eps, num_eps)
    eps_range = np.exp(log_eps_range)

    return eps_range

def find_Theiler_size(X):
    '''
    Returns 'optimal' theiler window size
    '''

    assert X.shape[1] == 3, 'Wrong dimensions'

    first_zeros = []
    
    for x_i in X.T:
        ac = autocorrelation(x_i)

        sc = sign_changes(ac)

        if len(sc) >= 1:
            first_zeros.append(sc[0])

    if len(first_zeros) >= 1:
        return min(first_zeros)
    
    else:
        return 0
    
def rescale_Theiler(sampling_rate, window_size=FIRST_ZERO):
    '''
    returns number of points the theiler window should be; assumes the original is calculated on the full dataset with dt=0.1
    '''

    new_window = np.round(window_size / sampling_rate)

    return int(new_window)

In [ ]:
from itertools import groupby

def longest_subsequence(arr):
    '''
    for the scaling region; finds the longest subsequence of 'True' values

    returns (starting index, length)
    '''

    ## rescale boolean array
    grouped_arr = groupby(arr)

    length = 0
    max_idx = 0

    idx = 0

    for val, group in grouped_arr:

        cur_length = len([i for i in group])

        ## only looking for True values
        if val and cur_length >= length:

                length = cur_length
                max_idx = idx

        idx += cur_length

    return max_idx, length

def subseq_mask(n, subseq_info):
    '''
    gives mask to get the longest subsequence

    subseq_info should be (starting index, length)
    '''

    base_mask = np.zeros(n).astype(bool)

    for i in range(subseq_info[0], subseq_info[0]+subseq_info[1]):
        base_mask[i] = True

    return base_mask

def find_corr_dim(mask, eps, corr_sum):

    scaling_eps = eps[mask]
    scaling_corr = np.array(corr_sum)[mask]

    D_Corr = np.polyfit(np.log(scaling_eps), np.log(scaling_corr), 1)[0]

    return D_Corr

In [ ]:
## correlation

t = soln.t
X = soln.y.T

SCALING_THRESHOLD = 0.01
num_eps = 20
num_trajectories = 50

eps_range = np.logspace(-4, -1, num_eps)

test_trajectory_end = np.floor(np.linspace(t_transient + 50, t_end, num_trajectories)).astype(int)

fig, axs = plt.subplots(1, 3, figsize=(15, 6))
axs = axs.flatten()

corr_dims = []
window_sizes = []

base_mask = t >= t_transient

for traj_id in tqdm(range(num_trajectories)):
    traj_end = test_trajectory_end[traj_id]

    traj_mask = (t <= traj_end) & base_mask

    X_sample = X[traj_mask]
    t_sample = t[traj_mask]

    corr_sums = []

    window = find_Theiler_size(X_sample)
    window_sizes.append(window)

    for eps_id in range(num_eps):

        eps = eps_range[eps_id]
        csum = correlation_sum(eps, X_sample, window, track=False)

        corr_sums.append(csum)

    axs[0].scatter(eps_range, corr_sums, s=5, alpha=0.4)

    second_diff = np.diff(np.diff(np.log(corr_sums)))
    scaling_mask = second_diff < SCALING_THRESHOLD

    ### LOOK FOR LONGEST CONTIGUOUS SUBSEQUENCE
    ssi = longest_subsequence(scaling_mask)

    if ssi[1] < 2: ## if no pairs of points
        D_Corr = 0

    else:

        final_mask = subseq_mask(len(eps_range), ssi)

        scaling_eps = eps_range[final_mask]
        scaling_corr = np.array(corr_sums)[final_mask]

        # print(f'eps: {scaling_eps} // dc: {scaling_corr}')

        D_Corr = np.polyfit(np.log(scaling_eps), np.log(scaling_corr), 1)[0]

    corr_dims.append(D_Corr)

axs[2].plot(test_trajectory_end, window_sizes)

axs[1].set_xlabel('Trajectory endpoint')
axs[1].set_ylabel('Correlation Dimension')
axs[1].plot(test_trajectory_end, corr_dims, marker='o')

axs[0].set_xlabel('Epsilon')
axs[0].set_ylabel('Correlation Sum')
axs[0].set_xscale('log')
axs[0].set_yscale('log')

In [ ]:
import matplotlib.ticker as ticker

## trajectory length

test_start = 0
test_end = max(soln.t) # max(test_trajectory_end)

## lyapunov
t = soln.t

t_mask = (t >= test_start) & (t <= test_end)

t = t[t_mask]
X = soln.y.T[t_mask]

le, hist = solve_le(X, t)

ky_dims = ky_dim(hist)

upper_mask = ky_dims < 3
lower_mask = ky_dims > 2

mask = upper_mask & lower_mask

final_ky_dims = ky_dims[mask]

plot_t = t[t >= t_transient][mask]

print(final_ky_dims[-1])

fig, ax = plt.subplots(figsize=(8, 8/np.sqrt(2)))

ax.set_ylim(1, 3)

ky_interval = 1

pt_size = 3

ax.scatter(test_trajectory_end, corr_dims, s=pt_size, label='$D_\\text{Corr}$', color='blue')
ax.scatter(plot_t[::ky_interval], final_ky_dims[::ky_interval], s=pt_size, label='$D_\\text{KY}$', color='red')

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))

ax.set_ylabel('Dimension')
ax.set_xlabel('Simulation End Time')

ax.axhline(2.06634548946088, zorder=2, color='black', linestyle=':', alpha=0.7)
ax.axhline(1.89, zorder=2, color='black', linestyle=':', alpha=0.7)

ax.legend(markerscale=1)

In [ ]:
import matplotlib.ticker as ticker

## trajectory length

test_start = 0
test_end = max(soln.t) # max(test_trajectory_end)

## lyapunov
t = soln.t

t_mask = (t >= test_start) & (t <= test_end)

t = t[t_mask]
X = soln.y.T[t_mask]

le, hist = solve_le(X, t)

ky_dims = ky_dim(hist)

upper_mask = ky_dims < 3
lower_mask = ky_dims > 2

mask = upper_mask & lower_mask

final_ky_dims = ky_dims[mask]

plot_t = t[t >= t_transient][mask]

print(final_ky_dims[-1])

fig, ax = plt.subplots(figsize=(15, 12))

ax.set_ylim(2, 3)

ky_interval = 1

pt_size = 3

#ax.scatter(test_trajectory_end, corr_dims, s=pt_size, label='$D_\\text{Corr}$', color='blue')
ax.scatter(plot_t[::ky_interval], final_ky_dims[::ky_interval], s=pt_size, label='$D_\\text{KY}$', color='red')

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))

ax.set_ylabel('Dimension')
ax.set_xlabel('Simulation End Time')

ax.axhline(2.06634548946088, zorder=2, color='black', linestyle=':', alpha=0.7)
#ax.axhline(1.89, zorder=2, color='black', linestyle=':', alpha=0.7)

ax.legend(markerscale=1)

## **Sampling Rate**

In [ ]:
t_start = 0
t_transient = 500
t_end = 4000
dt = 0.01 ## original was 0.1
t_eval = np.arange(t_start, t_end+dt, dt)
X_0 = [0.05, 0, 0]

soln = solve_ivp(backup_func, (t_start, t_end), X_0, t_eval=t_eval, method='Radau', rtol=1e-10, atol=1e-8)

In [ ]:
import time

sample_rate_choices = np.arange(10, 301, 5)
NUM_EPS = 20
SCALING_THRESHOLD = 0.01
MIN_PTS = 2

full_t = soln.t
full_X = soln.y.T

corr_dims = []
ky_dims = []

for SAMPLE_CHOICE in range(len(sample_rate_choices)):
    start_time = time.time()

    sample_rate = sample_rate_choices[SAMPLE_CHOICE]

    t = full_t[::sample_rate]
    X = full_X[::sample_rate]

    print(f'[{SAMPLE_CHOICE}] Sample rate of {t[1]-t[0]:.02f}')

    ## KY DIM

    final_lyap, lyap_history = solve_le(X, t, track=False)

    try:
        Dim_KY = ky_dim(final_lyap)
    except Exception as e:
        print(e)
        Dim_KY = 0

    ky_dims.append(Dim_KY)

    print(f'KY Dimension: {Dim_KY:.02f}; Time elapsed: {time.time() - start_time:.02f}s')

    ## CORRELATION DIM

    eps_range = get_eps_range(X, num_eps = NUM_EPS)
    # window_size = find_Theiler_size(X)
    window_size = rescale_Theiler(t[1]-t[0])

    corr_sum_hist = []

    for EPS_CHOICE in range(len(eps_range)):

        eps = eps_range[EPS_CHOICE]

        # print(f'- eps: [{eps}]')

        corr_sum = correlation_sum(eps, X, window_size, track=False)

        corr_sum_hist.append(corr_sum)

    ## calculating D_Corr

    second_diff = np.diff(np.diff(np.log(corr_sum_hist)))

    scaling_mask = second_diff < SCALING_THRESHOLD

    ### LOOK FOR LONGEST CONTIGUOUS SUBSEQUENCE
    ssi = longest_subsequence(scaling_mask)

    if ssi[1] < 2: ## if no pairs of points
        D_Corr = 0

    else:

        final_mask = subseq_mask(len(eps_range), ssi)

        scaling_eps = eps_range[final_mask]
        scaling_corr = np.array(corr_sum_hist)[final_mask]

        D_Corr = np.polyfit(np.log(scaling_eps), np.log(scaling_corr), 1)[0]

    corr_dims.append(D_Corr)

    print(f'Correlation Dimension: {D_Corr:.02f}; Time elapsed: {time.time() - start_time:.02f}s')

In [ ]:
with open('test_sums_2.txt', 'w') as outfile:

    outfile.write(repr(sample_rate_choices))
    outfile.write('\n')

    outfile.write(repr(corr_dims))
    outfile.write('\n')

    outfile.write(repr(ky_dims))
    outfile.write('\n')

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(8, 5))

pt_size = 20

corr_dims = np.array(corr_dims)

x_corr = sample_rate_choices[:len(corr_dims)][corr_dims > 0]*dt
y_corr = corr_dims[corr_dims > 0]

ax.scatter(x_corr, y_corr, s=pt_size, color='blue', label='$D_\\text{Corr}$', plotnonfinite=True)

mean_corr = np.polyfit(x_corr, y_corr, 1)

# ax.plot(x_corr, mean_corr[0]*x_corr + mean_corr[1])

ky_dims = np.array(ky_dims)

ax.scatter(sample_rate_choices[ky_dims>0]*dt, ky_dims[ky_dims>0], s=pt_size, color='red', label='$D_\\text{KY}$')

ax.set_xlabel('Timestep ($\\Delta t$)')
ax.set_ylabel('Dimension')

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))

ax.legend(loc='upper left')

# Bonus Plots
Other plots

## Correlation Dimension

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FROM_FILE = True
THEILER = True

def import_correlation_sums(FROM_FILE=FROM_FILE, THEILER=THEILER):

    if THEILER:
        filepath = 'correlation_sums.txt' 
    else:
        filepath = 'correlation_sums-NO_THEILER.txt'

    if FROM_FILE:
        
        with open(filepath, 'r') as corr_file:

            eps, corr = corr_file.read().split('\n')

            eps_range = np.array(eval(eps))
            corr_sums = np.array(eval(corr))

    else:
        eps_range = [np.float64(0.004472945488587768), np.float64(0.004765763771385856), np.float64(0.005077751200546132), np.float64(0.005410162670977284), np.float64(0.005764335228415282), np.float64(0.006141693447370463), np.float64(0.006543755161138205), np.float64(0.0069721375669207965), np.float64(0.0074285637306168726), np.float64(0.007914869517442406), np.float64(0.008433010976259994), np.float64(0.008985072207318172), np.float64(0.009573273745046818), np.float64(0.010199981489626484), np.float64(0.01086771622325671), np.float64(0.01157916374940031), np.float64(0.012337185695786198), np.float64(0.013144831024623379), np.float64(0.01400534829632313), np.float64(0.014922198736057256), np.float64(0.015899070155709558), np.float64(0.016939891787218213), np.float64(0.018048850086972674), np.float64(0.019230405574834453), np.float64(0.020489310775512783), np.float64(0.021830629334460002), np.float64(0.02325975638517584), np.float64(0.024782440249843156), np.float64(0.026404805560580814), np.float64(0.028133377894313374), np.float64(0.029975110020345495), np.float64(0.03193740986621564), np.float64(0.03402817031431505), np.float64(0.036255800949121765), np.float64(0.038629261882745396), np.float64(0.04115809979483774), np.float64(0.04385248633183125), np.float64(0.046723259019957095), np.float64(0.049781964856605625), np.float64(0.05304090675536514), np.float64(0.056513193031552965), np.float64(0.060212790127282555), np.float64(0.06415457878814086), np.float64(0.06835441391743263), np.float64(0.07282918834874183), np.float64(0.07759690079331942), np.float64(0.08267672823560004), np.float64(0.08808910306804096), np.float64(0.09385579527554012), np.float64(0.10000000000000002)]
        corr_sums = [np.float64(3.489857927839813e-05), np.float64(3.966300675222325e-05), np.float64(4.49725050633028e-05), np.float64(5.105288478992598e-05), np.float64(5.787628047996371e-05), np.float64(6.548169121439123e-05), np.float64(7.410176841447533e-05), np.float64(8.390098100780485e-05), np.float64(9.481503762239553e-05), np.float64(0.00010735752866579922), np.float64(0.00012162584514840757), np.float64(0.00013722679549907504), np.float64(0.0001547973814430711), np.float64(0.0001748499005313848), np.float64(0.00019728844289026927), np.float64(0.00022232304122714598), np.float64(0.0002503869280105892), np.float64(0.0002822366377138977), np.float64(0.00031811044505678594), np.float64(0.0003588822457636369), np.float64(0.00040494784222273254), np.float64(0.0004572476055793907), np.float64(0.0005162319268936543), np.float64(0.0005820080379481985), np.float64(0.0006559659352760506), np.float64(0.0007396047676497494), np.float64(0.0008343200168287097), np.float64(0.0009409719972674031), np.float64(0.0010603016036403207), np.float64(0.0011943496539637748), np.float64(0.0013453664717772698), np.float64(0.0015135883862396732), np.float64(0.0017019168310699001), np.float64(0.001912373783724062), np.float64(0.0021489626059287125), np.float64(0.0024144262372297787), np.float64(0.0027117735565129436), np.float64(0.003043715909926381), np.float64(0.0034142987962789873), np.float64(0.0038282838062913523), np.float64(0.00428957778932002), np.float64(0.004802143074585686), np.float64(0.005368270616469538), np.float64(0.0060040232289152815), np.float64(0.006714664695830097), np.float64(0.007502234956589916), np.float64(0.00836816532892975), np.float64(0.009317072239627003), np.float64(0.010358280134220355), np.float64(0.011502927411318042)]

    return eps_range, corr_sums

def mask_arr(arr, lower_bound=None, upper_bound=None):

    if lower_bound and upper_bound:
        assert lower_bound <= upper_bound

    mask = np.ones(arr.shape[0]).astype(bool)

    if lower_bound:

        lower_mask = arr >= lower_bound
        mask = mask & lower_mask       

    if upper_bound:
        
        upper_mask = arr <= upper_bound
        mask = mask & upper_mask

    masked_arr = arr[mask]

    return masked_arr, mask

def log_mask_arr(arr, lower_bound=None, upper_bound=None):

    if lower_bound:

        lower_bound = 10**lower_bound

    if upper_bound:

        upper_bound = 10**upper_bound

    return mask_arr(arr, lower_bound, upper_bound)

def fit_log_line(eps, corr, lower=None, upper=None):

    scaled_eps, scale_mask = mask_arr(eps, lower, upper)
    scaled_corr = corr[scale_mask]

    coeffs = np.polyfit(
        np.log(scaled_eps), np.log(scaled_corr), 1
    )

    return coeffs, scaled_eps

def plot_loglog(x, coeffs, ax=None, **plot_kwargs):
    '''
    plot a straight line on a log log graph
    '''

    if ax == None:
        fig, ax = plt.subplots()

    return ax.plot(x, x**coeffs[0] * np.exp(coeffs[1]), **plot_kwargs)

def plot_second_order_diff(eps_range, corr_sums, ax=None, **plot_kwargs):
    '''
    plots second order difference 
    '''

    if ax == None:
        fig, ax = plt.subplots()

    corr_diffs = np.diff(np.diff(np.log(corr_sums)))

    return ax.semilogx(eps_range[1:-1], corr_diffs, **plot_kwargs)

In [ ]:
THEILER_COLOR = 'blue'
NT_COLOR = 'red'

eps_range, corr_sums = import_correlation_sums()
NT_eps_range, NT_corr_sums = import_correlation_sums(THEILER=False)

coeffs, scaled_eps = fit_log_line(eps_range, corr_sums, 0.005, 0.1)
NT_coeffs, NT_scaled_eps = fit_log_line(NT_eps_range, NT_corr_sums, 0.005, 0.1)

fig, axs = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
axs = axs.ravel()

D_CORR = r'D_\text{Corr}'

ax = axs[0]

ax.scatter(eps_range, corr_sums, marker='o', s=1, label=f'With Theiler Window (${D_CORR}={coeffs[0]:.02f}$)', color=THEILER_COLOR, alpha=0.6)
ax.scatter(NT_eps_range, NT_corr_sums, marker='o', s=1, label=f'Without Theiler Window (${D_CORR}={NT_coeffs[0]:.02f}$)', color=NT_COLOR, alpha=0.6)

ax.set_yscale('log')
ax.set_xscale('log')

plot_loglog(scaled_eps, coeffs, ax=ax, color=THEILER_COLOR, linewidth=1.5)
plot_loglog(NT_scaled_eps, NT_coeffs, ax=ax, color=NT_COLOR)

ax.legend(markerscale=3)

ax.set_ylabel(r'Correlation Sum ($C(\varepsilon)$)')
#ax.set_xlabel(r'Ball Size ($\varepsilon$)')

print(f'WITH THEILER: {coeffs}')
print(f'WITHOUT THEILER: {NT_coeffs}')

ax = axs[1]

plot_second_order_diff(eps_range, corr_sums, color=THEILER_COLOR, ax=ax)
plot_second_order_diff(NT_eps_range, NT_corr_sums, color=NT_COLOR, ax=ax)

ax.set_ylabel('Second Order Difference')
ax.set_xlabel('Ball Size ($\\varepsilon$)')

ax.axvline(0.005, color='black', linestyle=':', alpha=0.6)
ax.axvline(0.1, color='black', linestyle=':', alpha=0.6)

fig.tight_layout()

ax.plot()